# Module 11 Instructor Colab Notebook: Expectation, Variance, Covariance, and Correlation

## A Mathematical Bridge to Artificial Intelligence

Probability distributions describe uncertainty, while expectation and variance summarize their center and spread. Covariance extends this idea to paired variables, and correlation removes measurement units to express standardized linear association.

These quantities appear directly in mean squared error, feature analysis, Gaussian models, minibatch statistics, and batch normalization. The notebook carefully separates population quantities from sample estimates and emphasizes that association does not establish causation.

**Recommended pacing:** three class meetings. The algebraic proofs can be presented selectively, but the bias-variance and batch-normalization calculations should be completed in full.


> **How to use this notebook in class**
>
> Ask students to predict signs and units before calculating. Use centered-data tables to make every covariance contribution visible. In batch normalization, distinguish the normalized minibatch values from the learned scale and shift, and distinguish training statistics from inference-time statistics.

## Notebook map

| Section | Main mathematical ideas | Direct AI connection |
|---|---|---|
| 1 | Expected value and linearity | Average loss and population objectives |
| 2 | Variance and standard deviation | Signal and error spread |
| 3 | Covariance | Joint feature variation |
| 4 | Correlation and standardization | Unit-free feature association |
| 5 | Expected squared error | Mean squared error and bias-variance identity |
| 6 | Batch normalization | Minibatch centering and rescaling |
| Integrated lab | Error and activation analysis | Combine MSE, covariance, and normalization |


In [ ]:
# Core setup: compatible with Google Colab and standard Jupyter.
import math
import heapq
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

try:
    import ipywidgets as widgets
    from ipywidgets import interact, FloatSlider, IntSlider
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

print("Setup complete.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)


In [ ]:
def reveal_markdown(button_text, solution_markdown):
    if not WIDGETS_AVAILABLE:
        display(Markdown("**Instructor solution**"))
        display(Markdown(solution_markdown))
        return

    button = widgets.Button(description=button_text, button_style="info")
    output = widgets.Output()
    state = {"open": False}

    def toggle(_):
        with output:
            output.clear_output()
            state["open"] = not state["open"]
            if state["open"]:
                display(Markdown(solution_markdown))
                button.description = "Hide instructor solution"
            else:
                button.description = button_text

    button.on_click(toggle)
    display(button, output)


def draw_graph(vertices, edges, positions, directed=False, weights=None,
               title="Graph", highlighted_edges=None, highlighted_vertices=None):
    highlighted_edges = set(highlighted_edges or [])
    highlighted_vertices = set(highlighted_vertices or [])
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.axis("off")
    for index, edge in enumerate(edges):
        u, v = edge
        color = "crimson" if edge in highlighted_edges or (v, u) in highlighted_edges else "#475569"
        arrowstyle = "->" if directed else "-"
        ax.annotate("", xy=positions[v], xytext=positions[u],
                    arrowprops=dict(arrowstyle=arrowstyle, linewidth=2.5,
                                    color=color, shrinkA=18, shrinkB=18))
        if weights is not None:
            midpoint = ((positions[u][0]+positions[v][0])/2,
                        (positions[u][1]+positions[v][1])/2)
            ax.text(midpoint[0], midpoint[1]+0.05, str(weights[index]),
                    ha="center", va="center",
                    bbox=dict(facecolor="white", edgecolor="none", pad=1))
    for vertex in vertices:
        color = "#fecaca" if vertex in highlighted_vertices else "#dbeafe"
        ax.text(positions[vertex][0], positions[vertex][1], str(vertex),
                ha="center", va="center", fontsize=12,
                bbox=dict(boxstyle="circle,pad=0.45", facecolor=color, edgecolor="#1d4ed8"))
    ax.set_title(title)
    plt.show()


## Module learning objectives

By the end of the module, students should be able to:

1. compute expectations for discrete and continuous variables;
2. apply linearity of expectation;
3. compute variance using both equivalent formulas;
4. interpret standard deviation in original units;
5. compute and interpret covariance;
6. standardize variables and compute correlation;
7. derive the expected squared-error decomposition; and
8. carry out batch normalization by hand.


---

# Section 1: Expected Value

## 1. Probability-weighted center

For a discrete random variable with PMF $$p_X,$$ the **expected value** is

$$
\boxed{E[X]=\sum_x x\,p_X(x),}
$$

provided the sum converges absolutely. For a continuous random variable with PDF $$f_X,$$

$$
\boxed{E[X]=\int_{-\infty}^{\infty}x f_X(x)\,dx,}
$$

provided the integral exists.

Expectation is a probability-weighted long-run center. It need not be one of the values the random variable can take.


## 2. Detailed discrete example

Suppose the number of incorrect labels in a tiny batch has PMF

| $$x$$ | 0 | 1 | 2 | 3 |
|---|---:|---:|---:|---:|
| $$P(X=x)$$ | 0.20 | 0.40 | 0.30 | 0.10 |

Then

$$
E[X]=0(0.20)+1(0.40)+2(0.30)+3(0.10).
$$

Compute each contribution:

$$
0(0.20)=0,
$$

$$
1(0.40)=0.40,
$$

$$
2(0.30)=0.60,
$$

$$
3(0.10)=0.30.
$$

Therefore

$$
\boxed{E[X]=0+0.40+0.60+0.30=1.30.}
$$

An average of $$1.30$$ errors does not mean a single batch can contain a fractional error.


## 3. Expected functions of a random variable

For a function $$g,$$

$$
E[g(X)]=\sum_x g(x)p_X(x)
$$

in the discrete case, and

$$
E[g(X)]=\int g(x)f_X(x)\,dx
$$

in the continuous case.

For the preceding distribution,

$$
E[X^2]
=0^2(0.20)+1^2(0.40)+2^2(0.30)+3^2(0.10).
$$

Thus

$$
E[X^2]=0+0.40+1.20+0.90=\boxed{2.50}.
$$


## 4. Linearity of expectation

For constants $$a,b$$ and random variables $$X,Y,$$

$$
\boxed{E[aX+bY]=aE[X]+bE[Y].}
$$

Independence is not required. In the discrete joint case,

$$
E[aX+bY]
=\sum_x\sum_y(ax+by)p_{X,Y}(x,y).
$$

Distribute the sum:

$$
=a\sum_x\sum_yx p_{X,Y}(x,y)
+b\sum_x\sum_yy p_{X,Y}(x,y).
$$

Marginalizing the unwanted variable gives

$$
=aE[X]+bE[Y].
$$


## 5. Expected loss

If $$Z$$ is a random data example and $$\ell(\theta;Z)$$ is its loss, the population objective is

$$
R(\theta)=E[\ell(\theta;Z)].
$$

A dataset average

$$
\widehat{R}_n(\theta)=\frac{1}{n}\sum_{i=1}^{n}\ell(\theta;z_i)
$$

estimates this expectation. The expectation is a property of the population model; the empirical mean is a random estimate based on observed data.


In [ ]:
x_expect11 = np.array([0, 1, 2, 3], dtype=float)
p_expect11 = np.array([0.20, 0.40, 0.30, 0.10])
mean_expect11 = np.sum(x_expect11*p_expect11)
second_moment_expect11 = np.sum((x_expect11**2)*p_expect11)
print("Expected value:", mean_expect11)
print("Second moment:", second_moment_expect11)


> **Common mistakes in Section 1**
>
> - Averaging support values without probability weights.
> - Assuming an expectation must be an observable value.
> - Confusing a population expectation with one sample average.
> - Believing linearity requires independence.
> - Omitting the PDF or PMF when computing an expectation.

## Section 1 terminology

| Term | Meaning |
|---|---|
| Expected value | Probability-weighted center of a random variable |
| Population mean | Expected value under the population distribution |
| Sample mean | Arithmetic average of observed data |
| Moment | Expected value of a power or other specified function |
| Linearity of expectation | Expectation distributes across linear combinations |
| Expected loss | Population average of a model's loss |
| Empirical risk | Observed sample average of loss |


## Section 1 guided worksheet

A random loss $$L$$ has values $$0,$$ $$1,$$ and $$4$$ with probabilities $$0.5,$$ $$0.3,$$ and $$0.2.$$

1. Verify that the probabilities form a PMF.
2. Compute $$E[L].$$
3. Compute $$E[L^2].$$
4. Compute $$E[3L+2]$$ using linearity.


In [ ]:
reveal_markdown("Open Section 1 solutions", r"""
1. The probabilities are nonnegative and

   $$
   0.5+0.3+0.2=1.
   $$

2. The expected loss is

   $$
   E[L]=0(0.5)+1(0.3)+4(0.2)
   =0+0.3+0.8
   =\boxed{1.1}.
   $$

3. The second moment is

   $$
   E[L^2]=0^2(0.5)+1^2(0.3)+4^2(0.2)
   =0+0.3+3.2
   =\boxed{3.5}.
   $$

4. By linearity,

   $$
   E[3L+2]=3E[L]+2
   =3(1.1)+2
   =\boxed{5.3}.
   $$
""")


---

# Section 2: Variance and Standard Deviation

## 1. Spread around the mean

If $$\mu=E[X],$$ the **variance** is

$$
\boxed{\operatorname{Var}(X)=E[(X-\mu)^2].}
$$

Squaring prevents positive and negative deviations from canceling and penalizes large deviations more strongly. Variance has squared units.

The **standard deviation** is

$$
\boxed{\sigma_X=\sqrt{\operatorname{Var}(X)}.}
$$

It returns to the original units of $$X.$$


## 2. Detailed variance calculation

For the error-count distribution from Section 1,

$$
\mu=1.30.
$$

Compute squared deviations and contributions:

| $$x$$ | $$x-1.3$$ | $$(x-1.3)^2$$ | Probability | Contribution |
|---:|---:|---:|---:|---:|
| 0 | -1.3 | 1.69 | 0.20 | 0.338 |
| 1 | -0.3 | 0.09 | 0.40 | 0.036 |
| 2 | 0.7 | 0.49 | 0.30 | 0.147 |
| 3 | 1.7 | 2.89 | 0.10 | 0.289 |

Add:

$$
\operatorname{Var}(X)=0.338+0.036+0.147+0.289
=\boxed{0.81}.
$$

Therefore

$$
\operatorname{SD}(X)=\sqrt{0.81}=\boxed{0.9}.
$$


## 3. Computational identity for variance

Expand the square:

$$
(X-\mu)^2=X^2-2\mu X+\mu^2.
$$

Take expectations:

$$
\operatorname{Var}(X)
=E[X^2]-2\mu E[X]+\mu^2.
$$

Since $$E[X]=\mu,$$

$$
\operatorname{Var}(X)
=E[X^2]-2\mu^2+\mu^2.
$$

Thus

$$
\boxed{\operatorname{Var}(X)=E[X^2]-E[X]^2.}
$$

For the example,

$$
2.50-(1.30)^2=2.50-1.69=0.81.
$$


## 4. Shifts and scales

For constants $$a,b,$$

$$
\boxed{\operatorname{Var}(aX+b)=a^2\operatorname{Var}(X).}
$$

Adding $$b$$ shifts every value and the mean equally, so spread does not change. Multiplying by $$a$$ multiplies deviations by $$a,$$ and squared deviations by $$a^2.$$

Consequently,

$$
\operatorname{SD}(aX+b)=|a|\operatorname{SD}(X).
$$


## 5. Population and sample variance

For a full population of $$N$$ values,

$$
\sigma^2=\frac{1}{N}\sum_{i=1}^{N}(x_i-\mu)^2.
$$

For a sample used to estimate an unknown population variance, the common unbiased estimator is

$$
s^2=\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\overline{x})^2.
$$

The denominator $$n-1$$ is called **Bessel's correction**. Batch normalization typically uses a minibatch variance convention tied to its implementation, which should not be silently confused with an unbiased statistical estimator.


In [ ]:
variance_expect11 = np.sum(((x_expect11-mean_expect11)**2)*p_expect11)
variance_identity11 = second_moment_expect11-mean_expect11**2
standard_deviation_expect11 = np.sqrt(variance_expect11)
print("Variance from deviations:", variance_expect11)
print("Variance from identity:", variance_identity11)
print("Standard deviation:", standard_deviation_expect11)


> **Common mistakes in Section 2**
>
> - Forgetting to square deviations.
> - Reporting variance in original rather than squared units.
> - Confusing variance with standard deviation.
> - Applying linearity of expectation incorrectly to variance.
> - Mixing population and sample denominators without explanation.
> - Assuming small variance automatically means an accurate model.

## Section 2 terminology

| Term | Meaning |
|---|---|
| Variance | Expected squared distance from the mean |
| Standard deviation | Square root of variance in original units |
| Squared deviation | Square of a value's difference from the mean |
| Population variance | Spread of the entire modeled population |
| Sample variance | Estimate of population variance from data |
| Bessel's correction | Use of denominator $$n-1$$ in unbiased sample variance |
| Scale transformation | Multiplication changing variance by the squared factor |


## Section 2 guided worksheet

A variable has

$$
E[X]=4
$$

and

$$
E[X^2]=22.
$$

1. Compute the variance.
2. Compute the standard deviation.
3. Find $$\operatorname{Var}(3X-7).$$
4. Find the standard deviation of $$3X-7.$$


In [ ]:
reveal_markdown("Open Section 2 solutions", r"""
1. Use the computational identity:

   $$
   \operatorname{Var}(X)=22-4^2=22-16=\boxed{6}.
   $$

2. The standard deviation is

   $$
   \boxed{\sqrt{6}}.
   $$

3. A shift does not change variance, while scaling by three multiplies variance by nine:

   $$
   \operatorname{Var}(3X-7)=3^2\operatorname{Var}(X)
   =9(6)=\boxed{54}.
   $$

4. The standard deviation is

   $$
   \sqrt{54}=3\sqrt{6}.
   $$

   Equivalently, scaling by three multiplies standard deviation by three.
""")


---

# Section 3: Covariance

## 1. Joint linear movement

For random variables $$X,Y$$ with means $$\mu_X,\mu_Y,$$ the **covariance** is

$$
\boxed{\operatorname{Cov}(X,Y)
=E[(X-\mu_X)(Y-\mu_Y)].}
$$

- Positive covariance means above-average values of one tend to occur with above-average values of the other.
- Negative covariance means above-average values of one tend to occur with below-average values of the other.
- Zero covariance means no linear co-movement, but nonlinear dependence can remain.

Covariance units are the product of the units of $$X$$ and $$Y.$$


## 2. Computational identity

Expand:

$$
(X-\mu_X)(Y-\mu_Y)
=XY-\mu_XY-\mu_YX+\mu_X\mu_Y.
$$

Take expectations:

$$
\operatorname{Cov}(X,Y)
=E[XY]-\mu_XE[Y]-\mu_YE[X]+\mu_X\mu_Y.
$$

Since $$E[X]=\mu_X$$ and $$E[Y]=\mu_Y,$$

$$
\boxed{\operatorname{Cov}(X,Y)=E[XY]-E[X]E[Y].}
$$

Also,

$$
\operatorname{Cov}(X,X)=\operatorname{Var}(X).
$$


## 3. Detailed paired-data calculation

Treat the three pairs

$$
(1,2),\quad(2,4),\quad(3,5)
$$

as an equally weighted population. The means are

$$
\mu_X=\frac{1+2+3}{3}=2
$$

and

$$
\mu_Y=\frac{2+4+5}{3}=\frac{11}{3}.
$$

The centered products are

$$
(1-2)\left(2-\frac{11}{3}\right)
=(-1)\left(-\frac{5}{3}\right)=\frac{5}{3},
$$

$$
(2-2)\left(4-\frac{11}{3}\right)=0,
$$

and

$$
(3-2)\left(5-\frac{11}{3}\right)
=(1)\left(\frac{4}{3}\right)=\frac{4}{3}.
$$

Average:

$$
\operatorname{Cov}(X,Y)
=\frac{1}{3}\left(\frac{5}{3}+0+\frac{4}{3}\right)
=\frac{1}{3}(3)
=\boxed{1}.
$$


## 4. Covariance matrix

For a random vector

$$
\mathbf{X}=
\begin{bmatrix}
X_1\\
\vdots\\
X_d
\end{bmatrix},
$$

the covariance matrix is

$$
\boxed{\Sigma=E[(\mathbf{X}-\boldsymbol{\mu})(\mathbf{X}-\boldsymbol{\mu})^T].}
$$

Its diagonal entries are variances and off-diagonal entries are covariances:

$$
\Sigma_{ij}=\operatorname{Cov}(X_i,X_j).
$$

The matrix is symmetric and positive semidefinite. For any vector $$\mathbf{a},$$

$$
\mathbf{a}^T\Sigma\mathbf{a}
=\operatorname{Var}(\mathbf{a}^T\mathbf{X})\geq 0.
$$


In [ ]:
x_cov11 = np.array([1.0, 2.0, 3.0])
y_cov11 = np.array([2.0, 4.0, 5.0])
centered_cov11 = np.column_stack([x_cov11-x_cov11.mean(), y_cov11-y_cov11.mean()])
covariance_matrix11 = centered_cov11.T @ centered_cov11 / len(x_cov11)
print("Population covariance matrix:")
print(covariance_matrix11)


> **Common mistakes in Section 3**
>
> - Computing covariance from unpaired observations.
> - Forgetting to center both variables.
> - Interpreting zero covariance as independence in general.
> - Ignoring covariance units when comparing feature pairs.
> - Confusing an off-diagonal covariance with a variance.

## Section 3 terminology

| Term | Meaning |
|---|---|
| Covariance | Expected product of centered variables |
| Positive covariance | Variables tend to deviate in the same direction |
| Negative covariance | Variables tend to deviate in opposite directions |
| Uncorrelated | Having covariance zero |
| Covariance matrix | Matrix of all feature variances and covariances |
| Positive semidefinite | Producing a nonnegative quadratic form |
| Paired data | Observations aligned by the same cases or units |


## Section 3 guided worksheet

Treat pairs

$$
(1,4),\quad(2,2),\quad(3,0)
$$

as an equally weighted population.

1. Compute both means.
2. Compute covariance from centered products.
3. Predict the sign before calculating.
4. Explain how the units of covariance differ from the units of either variable.


In [ ]:
reveal_markdown("Open Section 3 solutions", r"""
1. The means are

   $$
   \mu_X=2
   $$

   and

   $$
   \mu_Y=2.
   $$

2. Centered products are

   $$
   (1-2)(4-2)=(-1)(2)=-2,
   $$

   $$
   (2-2)(2-2)=0,
   $$

   and

   $$
   (3-2)(0-2)=(1)(-2)=-2.
   $$

   Average:

   $$
   \operatorname{Cov}(X,Y)=\frac{-2+0-2}{3}
   =\boxed{-\frac{4}{3}}.
   $$

3. The sign should be negative because $$Y$$ decreases as $$X$$ increases.

4. If $$X$$ and $$Y$$ have different units, covariance has their product units. This makes raw covariance magnitude scale-dependent.
""")


---

# Section 4: Correlation and Standardization

## 1. Unit-free linear association

For nonzero standard deviations, the **correlation coefficient** is

$$
\boxed{\rho_{X,Y}
=\frac{\operatorname{Cov}(X,Y)}{\sigma_X\sigma_Y}.}
$$

It satisfies

$$
-1\leq\rho_{X,Y}\leq 1.
$$

Correlation is unitless. Values near one indicate strong positive linear association, values near negative one indicate strong negative linear association, and values near zero indicate weak linear association.


## 2. Detailed correlation calculation

For

$$
X=(1,2,3)
$$

and

$$
Y=(2,4,5),
$$

we found

$$
\operatorname{Cov}(X,Y)=1.
$$

The population variances are

$$
\operatorname{Var}(X)=\frac{(-1)^2+0^2+1^2}{3}=\frac{2}{3}
$$

and

$$
\operatorname{Var}(Y)
=\frac{(-5/3)^2+(1/3)^2+(4/3)^2}{3}
=\frac{14}{9}.
$$

Therefore

$$
\rho_{X,Y}
=\frac{1}{\sqrt{2/3}\sqrt{14/9}}
=\sqrt{\frac{27}{28}}
\approx \boxed{0.981981}.
$$


## 3. Standard scores

The standardized variable is

$$
Z_X=\frac{X-\mu_X}{\sigma_X}.
$$

It has mean zero and variance one. Correlation is the expected product of standardized variables:

$$
\rho_{X,Y}=E[Z_XZ_Y].
$$

Standardization makes differently scaled features comparable, but it does not make their distributions Gaussian and does not remove outliers or nonlinear dependence.


## 4. Correlation is not causation

Correlation can arise from:

- a direct causal effect;
- reverse causation;
- a shared hidden cause;
- selection effects;
- trends over time; or
- coincidence.

Therefore a correlation coefficient alone cannot establish that changing one variable will change the other. Causal claims require assumptions and designs beyond this descriptive statistic.


In [ ]:
rng_corr11 = np.random.default_rng(11)
x_visual_corr11 = np.linspace(-2, 2, 120)
datasets_corr11 = [
    (x_visual_corr11, 1.5*x_visual_corr11+rng_corr11.normal(0, 0.25, 120), "strong positive"),
    (x_visual_corr11, -x_visual_corr11+rng_corr11.normal(0, 0.7, 120), "negative"),
    (x_visual_corr11, x_visual_corr11**2+rng_corr11.normal(0, 0.12, 120), "nonlinear, near-zero linear correlation"),
]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
correlations_visual11 = []
for ax, (x_data, y_data, title) in zip(axes, datasets_corr11):
    correlation = np.corrcoef(x_data, y_data)[0, 1]
    correlations_visual11.append(correlation)
    ax.scatter(x_data, y_data, s=12, alpha=0.7)
    ax.set_title(f"{title}\ncorrelation = {correlation:.3f}")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
plt.tight_layout()
plt.show()


> **Common mistakes in Section 4**
>
> - Interpreting correlation as a slope.
> - Claiming zero correlation means no relationship.
> - Claiming correlation proves causation.
> - Computing correlation when one variable has zero standard deviation.
> - Assuming standardization removes skewness or outliers.
> - Comparing sample correlations without considering sample size or selection.

## Section 4 terminology

| Term | Meaning |
|---|---|
| Correlation | Standardized covariance measuring linear association |
| Standard score | Centered value divided by standard deviation |
| Unit-free measure | Quantity unchanged by measurement units under positive scaling |
| Linear association | Tendency to follow an approximate straight-line relationship |
| Nonlinear dependence | Dependence not summarized adequately by a line |
| Confounder | Variable related to both studied variables and capable of distorting association |
| Causal effect | Change in an outcome produced by intervening on another variable |


## Section 4 guided worksheet

Suppose

$$
\operatorname{Cov}(X,Y)=-6,
$$

$$
\sigma_X=2,
$$

and

$$
\sigma_Y=4.
$$

1. Compute the correlation.
2. Interpret its sign and magnitude.
3. If $$X$$ is converted to $$X^*=10X,$$ state the new correlation with $$Y.$$
4. Explain why no causal conclusion follows.


In [ ]:
reveal_markdown("Open Section 4 solutions", r"""
1. Compute

   $$
   \rho_{X,Y}=\frac{-6}{(2)(4)}
   =\boxed{-0.75}.
   $$

2. The variables have a fairly strong negative linear association: above-average values of one tend to occur with below-average values of the other.

3. Positive rescaling changes covariance and standard deviation by the same factor, so the correlation remains

   $$
   \boxed{-0.75}.
   $$

4. The coefficient describes association only. Hidden variables, selection, reverse influence, or other mechanisms could produce it.
""")


---

# Section 5: Mean Squared Error and Bias-Variance Identity

## 1. Mean squared error

For prediction error

$$
E=\widehat{Y}-Y,
$$

the population mean squared error is

$$
\operatorname{MSE}=E[E^2].
$$

For observed errors $$e_1,\ldots,e_n,$$

$$
\widehat{\operatorname{MSE}}
=\frac{1}{n}\sum_{i=1}^{n}e_i^2.
$$

MSE is an expectation of squared errors, not the square of the mean error.


## 2. Error variance plus squared bias

Let

$$
b=E[E]
$$

be the mean error, or bias under this sign convention. Using the variance identity,

$$
\operatorname{Var}(E)=E[E^2]-E[E]^2.
$$

Rearrange:

$$
\boxed{E[E^2]=\operatorname{Var}(E)+b^2.}
$$

Thus MSE combines random spread and systematic offset.

For errors

$$
-1,1,-2,0,
$$

the mean error is

$$
b=\frac{-1+1-2+0}{4}=-0.5.
$$

The MSE is

$$
\frac{1+1+4+0}{4}=1.5.
$$

Therefore error variance is

$$
1.5-(-0.5)^2=1.5-0.25=\boxed{1.25}.
$$


## 3. Best constant predictor under squared loss

For a random target $$Y$$ and constant prediction $$a,$$

$$
E[(Y-a)^2]
=\operatorname{Var}(Y)+(E[Y]-a)^2.
$$

The variance term does not depend on $$a.$$ The squared term is minimized when

$$
a=E[Y].
$$

Therefore the mean is the optimal constant prediction under expected squared loss.

Different losses produce different optimal summaries. Expected absolute loss, for example, is minimized by a median under suitable conditions.


## 4. Model bias-variance caution

The error identity above concerns one prediction error random variable. A broader supervised-learning **bias-variance decomposition** averages over varying training datasets and can include irreducible noise. Its exact form depends on the prediction target, loss, and data-generating assumptions.

Students should not use the words “bias” and “variance” without stating which random quantity varies and which expectation is being taken.


> **Common mistakes in Section 5**
>
> - Squaring mean error instead of averaging squared errors.
> - Forgetting that negative and positive errors both contribute positively to MSE.
> - Calling MSE a standard deviation.
> - Applying one bias-variance formula without defining the random sources.
> - Assuming the mean is optimal under every loss.

## Section 5 terminology

| Term | Meaning |
|---|---|
| Prediction error | Difference between prediction and target |
| Mean squared error | Expected or average squared prediction error |
| Mean error | Average signed prediction error |
| Squared bias | Square of the mean error in the stated convention |
| Error variance | Spread of prediction errors around their mean |
| Irreducible noise | Outcome variation not predictable from available information |
| Constant predictor | Model returning the same value for every input |


## Section 5 guided worksheet

Prediction errors are

$$
2,-1,1,-2.
$$

1. Compute mean error.
2. Compute MSE.
3. Compute population variance of error.
4. Verify MSE equals variance plus squared mean error.


In [ ]:
reveal_markdown("Open Section 5 solutions", r"""
1. The mean error is

   $$
   \frac{2-1+1-2}{4}=\boxed{0}.
   $$

2. The MSE is

   $$
   \frac{2^2+(-1)^2+1^2+(-2)^2}{4}
   =\frac{4+1+1+4}{4}
   =\boxed{2.5}.
   $$

3. Because the mean is zero, the squared deviations are the same squared errors:

   $$
   \operatorname{Var}(E)=\boxed{2.5}.
   $$

4. We verify

   $$
   \operatorname{Var}(E)+E[E]^2
   =2.5+0^2
   =2.5
   =\operatorname{MSE}.
   $$
""")


---

# Section 6: Batch Normalization

## 1. Why normalize activations?

During training, activation scales can differ across features and change as earlier layers update. **Batch normalization** standardizes each feature over a minibatch and then applies a learned scale and shift.

For one feature with batch values $$x_1,\ldots,x_m,$$ compute

$$
\mu_B=\frac{1}{m}\sum_{i=1}^{m}x_i
$$

and

$$
\sigma_B^2=\frac{1}{m}\sum_{i=1}^{m}(x_i-\mu_B)^2.
$$

Then

$$
\widehat{x}_i=\frac{x_i-\mu_B}{\sqrt{\sigma_B^2+\varepsilon}},
$$

followed by

$$
y_i=\gamma\widehat{x}_i+\beta.
$$


## 2. Meaning of the parameters

The small constant $$\varepsilon>0$$ prevents division by zero and improves numerical stability. The learned parameters have different roles:

- $$\gamma$$ restores or changes scale;
- $$\beta$$ restores or changes location.

Without $$\gamma$$ and $$\beta,$$ normalization would force every feature to stay at one fixed scale and center. Learned affine parameters let the network select an appropriate representation.


## 3. Fully worked batch calculation

Use batch

$$
2,4,4,6.
$$

The batch mean is

$$
\mu_B=\frac{2+4+4+6}{4}=4.
$$

The variance is

$$
\sigma_B^2
=\frac{(2-4)^2+(4-4)^2+(4-4)^2+(6-4)^2}{4}.
$$

Therefore

$$
\sigma_B^2=\frac{4+0+0+4}{4}=2.
$$

Ignoring a tiny $$\varepsilon$$ for hand calculation, normalized values are

$$
\frac{2-4}{\sqrt{2}}=-\sqrt{2},
$$

$$
\frac{4-4}{\sqrt{2}}=0,
$$

$$
\frac{4-4}{\sqrt{2}}=0,
$$

and

$$
\frac{6-4}{\sqrt{2}}=\sqrt{2}.
$$

With $$\gamma=1.5$$ and $$\beta=0.5,$$

$$
y=1.5\widehat{x}+0.5
$$

gives approximately

$$
\boxed{(-1.6213,0.5,0.5,2.6213).}
$$


In [ ]:
batch_bn11 = np.array([2.0, 4.0, 4.0, 6.0])
mean_bn11 = batch_bn11.mean()
variance_bn11 = np.mean((batch_bn11-mean_bn11)**2)
normalized_bn11 = (batch_bn11-mean_bn11)/np.sqrt(variance_bn11)
gamma_bn11, beta_bn11 = 1.5, 0.5
output_bn11 = gamma_bn11*normalized_bn11+beta_bn11
print("Batch mean:", mean_bn11)
print("Batch variance:", variance_bn11)
print("Normalized values:", normalized_bn11)
print("Scaled and shifted values:", output_bn11)


## 4. Training and inference

During training, batch normalization uses current minibatch statistics. During inference, a model typically uses running estimates accumulated during training so one prediction does not depend on the other examples in an arbitrary inference batch.

Important limitations include:

- very small batches can produce noisy statistics;
- training and inference distributions can differ;
- axis choice depends on architecture and tensor layout; and
- normalization does not guarantee generalization or eliminate all optimization problems.


> **Common mistakes in Section 6**
>
> - Using sample variance denominator $$m-1$$ without checking the batch-normalization convention.
> - Normalizing across the wrong axis.
> - Forgetting $$\varepsilon.$$
> - Omitting learned scale and shift.
> - Using arbitrary inference-batch statistics instead of trained running statistics.
> - Assuming batch normalization makes every activation Gaussian.

## Section 6 terminology

| Term | Meaning |
|---|---|
| Batch normalization | Minibatch standardization followed by learned affine transformation |
| Minibatch mean | Feature average over the current minibatch |
| Minibatch variance | Feature spread over the current minibatch |
| Numerical epsilon | Small positive stabilizer in the denominator |
| Learned scale | Parameter $$\gamma$$ multiplying normalized values |
| Learned shift | Parameter $$\beta$$ added after normalization |
| Running statistics | Training-time estimates used during inference |
| Normalization axis | Dimension across which statistics are computed |


## Section 6 guided worksheet

Normalize batch

$$
1,3,5,7
$$

using population-style batch variance, ignoring $$\varepsilon$$ for hand calculation.

1. Compute mean and variance.
2. Compute normalized values.
3. Apply $$\gamma=2$$ and $$\beta=-1.$$
4. Verify the normalized mean is zero and variance is one.


In [ ]:
reveal_markdown("Open Section 6 solutions", r"""
The mean is

$$
\mu_B=\frac{1+3+5+7}{4}=4.
$$

The variance is

$$
\sigma_B^2
=\frac{(-3)^2+(-1)^2+1^2+3^2}{4}
=\frac{20}{4}=5.
$$

The normalized values are

$$
\boxed{\left(-\frac{3}{\sqrt{5}},-\frac{1}{\sqrt{5}},
\frac{1}{\sqrt{5}},\frac{3}{\sqrt{5}}\right)}.
$$

Applying scale and shift gives

$$
\boxed{\left(-\frac{6}{\sqrt{5}}-1,-\frac{2}{\sqrt{5}}-1,
\frac{2}{\sqrt{5}}-1,\frac{6}{\sqrt{5}}-1\right)}.
$$

The centered numerators sum to zero, so normalized mean is zero. Their squared average is

$$
\frac{9/5+1/5+1/5+9/5}{4}
=\frac{20/5}{4}=1.
$$
""")


---

# Integrated Instructor Lab: Analyze Errors and Normalize Two Features

## Part A: prediction errors

Use errors

$$
\mathbf{e}=(-1,1,-2,0).
$$

The mean error is

$$
\overline{e}=-0.5.
$$

The MSE is

$$
\frac{1+1+4+0}{4}=1.5.
$$

The population error variance is

$$
1.5-(-0.5)^2=1.25.
$$

This verifies

$$
\operatorname{MSE}=\operatorname{Var}(E)+E[E]^2.
$$

## Part B: paired activations

Use activation matrix

$$
A=
\begin{bmatrix}
1&2\\
2&1\\
3&4\\
4&3
\end{bmatrix}.
$$

Each column mean is $$2.5.$$ Centering gives

$$
A_c=
\begin{bmatrix}
-1.5&-0.5\\
-0.5&-1.5\\
0.5&1.5\\
1.5&0.5
\end{bmatrix}.
$$

The population covariance matrix is

$$
\Sigma=\frac{1}{4}A_c^TA_c
=
\begin{bmatrix}
1.25&0.75\\
0.75&1.25
\end{bmatrix}.
$$

Both standard deviations equal $$\sqrt{1.25},$$ so correlation is

$$
\rho=\frac{0.75}{1.25}=\boxed{0.6}.
$$


In [ ]:
errors_lab11 = np.array([-1.0, 1.0, -2.0, 0.0])
mean_error_lab11 = errors_lab11.mean()
mse_lab11 = np.mean(errors_lab11**2)
variance_error_lab11 = np.var(errors_lab11)

activations_lab11 = np.array([[1.0, 2.0], [2.0, 1.0], [3.0, 4.0], [4.0, 3.0]])
means_lab11 = activations_lab11.mean(axis=0)
centered_lab11 = activations_lab11-means_lab11
covariance_lab11 = centered_lab11.T @ centered_lab11/len(activations_lab11)
standardized_lab11 = centered_lab11/activations_lab11.std(axis=0)
correlation_lab11 = standardized_lab11.T @ standardized_lab11/len(activations_lab11)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(np.arange(len(errors_lab11)), errors_lab11, color="steelblue")
axes[0].axhline(mean_error_lab11, color="crimson", linestyle="--", label="mean error")
axes[0].set_xlabel("example")
axes[0].set_ylabel("prediction error")
axes[0].set_title("Signed errors and their mean")
axes[0].legend()
axes[1].scatter(activations_lab11[:, 0], activations_lab11[:, 1], s=80)
axes[1].set_xlabel("feature one")
axes[1].set_ylabel("feature two")
axes[1].set_title("Positive covariance between activations")
plt.tight_layout()
plt.show()

print("Mean error, MSE, variance:", mean_error_lab11, mse_lab11, variance_error_lab11)
print("Covariance matrix:")
print(covariance_lab11)
print("Correlation matrix:")
print(correlation_lab11)


## Integrated lab tasks

1. Recompute the error variance directly from centered errors.
2. Verify every entry of the covariance matrix by hand.
3. Standardize both activation columns and verify mean zero and variance one.
4. Explain why batch normalization standardizes each feature but does not generally remove correlation between features.
5. Explain why low MSE does not automatically imply unbiased errors.


In [ ]:
reveal_markdown("Open integrated-lab solution", r"""
Centered errors are

$$
-0.5,1.5,-1.5,0.5.
$$

Their squared average is

$$
\frac{0.25+2.25+2.25+0.25}{4}
=\frac{5}{4}=1.25.
$$

For the covariance matrix, first-column squared centered values sum to

$$
2.25+0.25+0.25+2.25=5,
$$

giving $$5/4=1.25.$$ The second column has the same variance. Cross-products are

$$
0.75,0.75,0.75,0.75,
$$

which average to $$0.75.$$

Standardizing divides both columns by $$\sqrt{1.25}.$$ Each standardized column has mean zero and variance one. Its covariance matrix becomes the correlation matrix

$$
\begin{bmatrix}
1&0.6\\
0.6&1
\end{bmatrix}.
$$

Batch normalization rescales columns independently, so the off-diagonal correlation can remain. Low MSE combines variance and squared mean error; it can be small while mean error is nonzero.
""")


---

# Module 11 Summary

1. Expectation is a probability-weighted population center.
2. Variance is expected squared distance from the mean; standard deviation restores original units.
3. Covariance measures linear co-movement in product units.
4. Correlation standardizes covariance into a unit-free coefficient.
5. Zero correlation does not rule out nonlinear dependence, and correlation does not prove causation.
6. MSE equals error variance plus squared mean error.
7. Batch normalization uses minibatch mean and variance, then learned scale and shift.

## Exit ticket

1. Why does linearity of expectation not require independence?
2. Why is standard deviation easier to interpret dimensionally than variance?
3. What does a negative covariance mean?
4. Why can zero correlation coexist with a strong nonlinear pattern?
5. What distinct roles do $$\gamma,$$ $$\beta,$$ and $$\varepsilon$$ play in batch normalization?
